# H5: Data-Reuploading with Trainable Scaling vs Fixed Encoding

**Hypothesis**: L-layer data re-uploading with per-layer trainable scaling parameters closes the accuracy gap to classical MLP classifiers on small benchmark suites.

**References**: arXiv:2403.07059, arXiv:2104.00021

In [ ]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
qml.numpy.random.seed(42)

In [ ]:
N_EPOCHS = 8
LR = 0.5

dev = qml.device("lightning.qubit", wires=1)

@qml.qnode(dev)
def reupload_circuit(params, x, dm_label):
    for p in params:
        qml.Rot(*x, wires=0)
        qml.Rot(*p, wires=0)
    return qml.expval(qml.Hermitian(dm_label, wires=[0]))

label_0 = np.array([[1], [0]])
label_1 = np.array([[0], [1]])
dm_labels = [l @ l.conj().T for l in [label_0, label_1]]

def train_reupload(X_train, y_train, n_layers=3):
    params = np.random.uniform(size=(n_layers, 3), requires_grad=True)
    opt = AdamOptimizer(LR, beta1=0.9, beta2=0.999)
    for _ in range(N_EPOCHS):
        for i in range(len(X_train)):
            dm = dm_labels[int(y_train[i])]
            params, _, _ = opt.step(
                lambda p, x, dm: (1 - reupload_circuit(p, x, dm)) ** 2,
                params, X_train[i], dm
            )
    return params

def test_reupload(params, X_test, y_test):
    preds = []
    for i in range(len(X_test)):
        fids = [reupload_circuit(params, X_test[i], dm) for dm in dm_labels]
        preds.append(int(np.argmax(fids)))
    return float(np.mean(np.array(preds) == y_test))

def load_benchmark(name):
    if name == "moons":
        X, y = make_moons(n_samples=200, noise=0.1, random_state=42)
    elif name == "circles":
        X, y = make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=42)
    else:
        X, y = make_classification(n_samples=200, n_features=4, n_informative=2,
                                    n_redundant=0, n_clusters_per_class=1, random_state=42)
    X = StandardScaler().fit_transform(X)
    if X.shape[1] < 3:
        X = np.pad(X, ((0, 0), (0, 3 - X.shape[1])), mode="constant")
    return X, y

In [ ]:
BENCHMARKS = ["moons", "circles", "blobs"]
LAYER_COUNTS = [1, 2, 3, 5]

results = {}
for bname in BENCHMARKS:
    print(f"\n--- {bname} ---")
    X, y = load_benchmark(bname)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    results[bname] = {}
    for L in LAYER_COUNTS:
        params = train_reupload(X_train, y_train, n_layers=L)
        acc = test_reupload(params, X_test, y_test)
        results[bname][f"reupload(L={L})"] = acc
        print(f"  L={L}: test_acc={acc:.4f}")
    
    mlp = MLPClassifier(early_stopping=True, max_iter=200, random_state=42)
    mlp.fit(X_train[:, :2] if X_train.shape[1] > 2 else X_train, y_train)
    mlp_acc = mlp.score(X_test[:, :2] if X_test.shape[1] > 2 else X_test, y_test)
    results[bname]["MLP"] = float(mlp_acc)
    print(f"  MLP: test_acc={mlp_acc:.4f}")

os.makedirs('results/H5', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'results/H5/run_{ts}.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to results/H5/run_{ts}.json")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(BENCHMARKS), figsize=(12, 4))
for idx, bname in enumerate(BENCHMARKS):
    ax = axes[idx]
    ls = [str(L) for L in LAYER_COUNTS] + ['MLP']
    vals = [results[bname].get(f"reupload(L={L})", 0) for L in LAYER_COUNTS]
    vals.append(results[bname].get("MLP", 0))
    colors_bar = ['C0'] * len(LAYER_COUNTS) + ['C3']
    ax.bar(ls, vals, color=colors_bar)
    ax.set_title(bname)
    ax.set_ylabel('Test Accuracy')
    ax.grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)

plt.suptitle('H5: Data-Reuploading vs Classical MLP', fontsize=14)
plt.tight_layout()
os.makedirs('results/H5', exist_ok=True)
plt.savefig('results/H5/plot.png', dpi=120)
plt.show()
print("Comparison complete: see bar charts for gap to MLP")